In [6]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Lenovo\Desktop\CLASSROOM\DA\projects\churn_analysis\Bank Customer Churn Prediction.csv")
print(df.shape)
df.head()

(10000, 12)


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
print(df['churn'].value_counts())
print("Churn rate:", round(df['churn'].mean() * 100, 2), "%")

churn
0    7963
1    2037
Name: count, dtype: int64
Churn rate: 20.37 %


In [8]:
for col in ['country', 'gender', 'active_member', 'products_number', 'credit_card']:
    print(f"--- Churn rate by {col} ---")
    print((df.groupby(col)['churn'].mean() * 100).round(1))
    print()

--- Churn rate by country ---
country
France     16.2
Germany    32.4
Spain      16.7
Name: churn, dtype: float64

--- Churn rate by gender ---
gender
Female    25.1
Male      16.5
Name: churn, dtype: float64

--- Churn rate by active_member ---
active_member
0    26.9
1    14.3
Name: churn, dtype: float64

--- Churn rate by products_number ---
products_number
1     27.7
2      7.6
3     82.7
4    100.0
Name: churn, dtype: float64

--- Churn rate by credit_card ---
credit_card
0    20.8
1    20.2
Name: churn, dtype: float64



In [9]:
print(df['products_number'].value_counts())
print()
# churn rate AND count together
print(df.groupby('products_number')['churn'].agg(['mean', 'count']))

products_number
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64

                     mean  count
products_number                 
1                0.277144   5084
2                0.075817   4590
3                0.827068    266
4                1.000000     60


In [11]:
# High-risk profile: churn rate by combinations of the strongest factors
profile = df.groupby(['country', 'active_member', 'products_number'])['churn'].agg(['mean', 'count'])
profile = profile[profile['count'] >= 30]   # only groups big enough to trust
profile = profile.sort_values('mean', ascending=False)
print((profile['mean'] * 100).round(1).head(10))
print()
print("Group sizes for those:")
print(profile['count'].head(10))

country  active_member  products_number
Germany  0              3                  92.7
France   0              3                  86.7
Germany  1              3                  85.4
Spain    0              3                  84.2
France   1              3                  68.2
Germany  0              1                  52.1
Spain    0              1                  32.5
Germany  1              1                  32.3
France   0              1                  29.6
Germany  0              2                  16.5
Name: mean, dtype: float64

Group sizes for those:
country  active_member  products_number
Germany  0              3                    55
France   0              3                    60
Germany  1              3                    41
Spain    0              3                    38
France   1              3                    44
Germany  0              1                   720
Spain    0              1                   548
Germany  1              1                   629
Franc

## Churn Diagnostic - Key Findings

**Overall churn rate:** 20.37% (2,037 of 10,000 customers)

**Strongest single drivers:**
- **Products:** 2 products = lowest churn (7.6%); 1 product = 27.7%; 3 products = 82.7% (266 customers); 4 products = 100% (only 60 customers - directional).
- **Geography:** Germany churns 32.4% vs ~16% for France/Spain.
- **Activity:** inactive members churn 26.9% vs 14.3% active.
- **Gender:** women 25.1% vs men 16.5%.
- **Credit card:** no meaningful effect (20.8% vs 20.2%).

**Highest-risk actionable segment:** inactive, single-product German customers churn at 52.1% - and there are 720 of them, making this the largest high-severity group to target.

**Key business insight:** moving single-product customers to a second product is the biggest retention lever (27.7% -> 7.6%), but pushing 3+ products coincides with severe churn - so cross-selling a *second* product helps, over-selling backfires.